# Example 05 - Pairwise MFEP Paths and CTMC Rates

This notebook mirrors `examples/05_pairwise_mfep_paths.py`. It is
the most detailed 2D example in the repository:

1. Detect all basins on the synthetic 2D FES.
2. Compute the MFEP for every basin pair.
3. Plot every path on the 2D surface and each 1D arc-length profile.
4. Assemble the full CTMC rate matrix and summarize pairwise rates.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "stochkin").is_dir():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "examples" / "data"
OUT_DIR = ROOT / "notebooks" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {ROOT}")
print(f"Notebook output directory: {OUT_DIR}")


In [ ]:
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import stochkin as sk

from stochkin.plotting import _apply_pub_axes, _apply_pub_cbar
from stochkin.style import publication_style


In [ ]:
_PAIR_COLOURS = [
    "#e6194b",
    "#3cb44b",
    "#4363d8",
    "#f58231",
    "#911eb4",
    "#42d4f4",
    "#f032e6",
    "#bfef45",
    "#fabebe",
    "#469990",
]


def colour_for(index):
    return _PAIR_COLOURS[index % len(_PAIR_COLOURS)]


In [ ]:
fes2d_path = DATA_DIR / "synthetic_2d_fes.dat"
D_s = 0.04
T = 300.0
neb_images = 120
neb_steps = 8000
max_basins = None
core_fraction = 0.05

result = sk.run_multi_mfep_ctmc(
    fes2d_path=fes2d_path,
    D_s=D_s,
    T=T,
    neb_images=neb_images,
    neb_steps=neb_steps,
    max_basins=max_basins,
    core_fraction=core_fraction,
)

basin_net = result["basin_network"]
basins = basin_net.basins
basin_labels = [f"B{bid}" for bid in result["basin_ids"]]
pair_keys = sorted(result["legs"].keys())


In [ ]:
K_ns = pd.DataFrame(
    result["K_ps"] * 1000.0,
    index=basin_labels,
    columns=basin_labels,
)
exit_times_ns = pd.Series(
    result["exit_ps"] / 1000.0,
    index=basin_labels,
    name="exit_time_ns",
)

print(f"Detected {len(basins)} basins")
for basin in basins:
    print(
        f"Basin {basin.id}: minimum ({basin.minimum[0]:.2f}, {basin.minimum[1]:.2f}), "
        f"F_min = {basin.f_min:.2f} kJ/mol"
    )


In [ ]:
K_ns


In [ ]:
rows = []
for i, j in pair_keys:
    leg = result["legs"][(i, j)]
    path = leg["mfep_path"]
    barrier = float(np.nanmax(path.F - np.nanmin(path.F)))
    K_leg = leg["K"]
    if K_leg.shape[0] >= 2:
        k_fwd = float(K_leg[0, -1])
        k_bwd = float(K_leg[-1, 0])
    else:
        k_fwd = float("nan")
        k_bwd = float("nan")
    rows.append(
        {
            "pair": f"B{basins[i].id}->B{basins[j].id}",
            "k_fwd": k_fwd,
            "k_bwd": k_bwd,
            "barrier_kj_mol": barrier,
        }
    )

summary_df = pd.DataFrame(rows)
summary_df


In [ ]:
x_grid, y_grid, fes_grid = sk.load_plumed_fes_2d(fes2d_path, verbose=False)
kT = result["kT"]

out_paths = OUT_DIR / "05_all_paths_on_fes.png"
out_profiles = OUT_DIR / "05_pairwise_profiles.png"
out_rates = OUT_DIR / "05_rate_matrix.png"
out_exit = OUT_DIR / "05_exit_times.png"

with publication_style():
    fig, ax = plt.subplots(figsize=(4.5, 3.6))
    cs = ax.contourf(
        x_grid,
        y_grid,
        (fes_grid / kT).T,
        levels=np.linspace(0, 15, 30),
        cmap="rainbow_r",
    )
    cbar = fig.colorbar(cs, ax=ax)
    _apply_pub_cbar(cbar, label=r"$F\,/\,k_\mathrm{B}T$")

    for basin in basins:
        ax.plot(
            *basin.minimum,
            "o",
            color="white",
            ms=10,
            zorder=5,
            markeredgecolor="black",
            markeredgewidth=0.6,
        )
        ax.text(
            basin.minimum[0] + 0.15,
            basin.minimum[1] + 0.15,
            f"B{basin.id}",
            color="white",
            fontsize=10,
            fontweight="bold",
            zorder=6,
            path_effects=[pe.withStroke(linewidth=2, foreground="black")],
        )

    for index, (i, j) in enumerate(pair_keys):
        path = result["legs"][(i, j)]["mfep_path"]
        colour = colour_for(index)
        ax.plot(
            path.x,
            path.y,
            "-",
            color=colour,
            lw=2.0,
            alpha=0.9,
            label=f"B{basins[i].id}->B{basins[j].id}",
            zorder=4,
        )

    ax.legend(fontsize=7, loc="upper left", framealpha=0.85)
    _apply_pub_axes(ax, r"CV$_1$", r"CV$_2$", "Pairwise MFEPs on 2D FES")
    fig.tight_layout()
    fig.savefig(out_paths, dpi=300)

n_pairs = len(pair_keys)
n_cols = min(n_pairs, 3)
n_rows = int(np.ceil(n_pairs / n_cols))

with publication_style():
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(3.3 * n_cols, 2.6 * n_rows),
        squeeze=False,
    )

    for index, (i, j) in enumerate(pair_keys):
        row, col = divmod(index, n_cols)
        ax = axes[row, col]
        leg = result["legs"][(i, j)]
        path = leg["mfep_path"]

        s = path.s
        F = path.F - np.nanmin(path.F)
        colour = colour_for(index)

        ax.fill_between(s, F / kT, alpha=0.20, color=colour)
        ax.plot(s, F / kT, "-", color=colour, lw=1.5)

        basin_network_1d = leg.get("basin_network")
        if basin_network_1d is not None:
            for basin_1d in basin_network_1d.basins:
                idx_min = np.argmin(np.abs(s - basin_1d.minimum))
                ax.axvline(s[idx_min], ls=":", color="grey", lw=0.7, alpha=0.6)

        K_leg = leg["K"]
        if K_leg.shape[0] >= 2:
            k_fwd = K_leg[0, -1]
            k_bwd = K_leg[-1, 0]
            ax.text(
                0.97,
                0.95,
                f"$k_{{->}}={k_fwd:.2e}$\n$k_{{<-}}={k_bwd:.2e}$",
                transform=ax.transAxes,
                fontsize=6,
                va="top",
                ha="right",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
            )

        _apply_pub_axes(
            ax,
            xlabel="Arc-length  s",
            ylabel=r"$F(s) / k_\mathrm{B}T$",
            title=f"B{basins[i].id} -> B{basins[j].id}",
        )

    for index in range(n_pairs, n_rows * n_cols):
        row, col = divmod(index, n_cols)
        axes[row, col].set_visible(False)

    fig.tight_layout()
    fig.savefig(out_profiles, dpi=300)

with publication_style():
    K_abs = np.abs(K_ns.to_numpy())
    with np.errstate(divide="ignore", invalid="ignore"):
        Klog = np.where(K_abs > 0, np.log10(K_abs), np.nan)

    fig, ax = plt.subplots(figsize=(3.3, 2.8))
    im = ax.imshow(Klog, cmap="magma_r", aspect="auto")
    ax.set_xticks(range(len(basin_labels)))
    ax.set_xticklabels(basin_labels)
    ax.set_yticks(range(len(basin_labels)))
    ax.set_yticklabels(basin_labels)

    finite_klog = Klog[np.isfinite(Klog)]
    threshold = np.nanmedian(finite_klog) if finite_klog.size else 0.0
    for ii in range(len(basin_labels)):
        for jj in range(len(basin_labels)):
            value = K_ns.to_numpy()[ii, jj]
            if ii != jj and value > 0:
                ax.text(
                    jj,
                    ii,
                    f"{value:.1e}",
                    ha="center",
                    va="center",
                    fontsize=5,
                    color="white" if Klog[ii, jj] < threshold else "black",
                )

    cbar = fig.colorbar(im, ax=ax)
    _apply_pub_cbar(cbar, label=r"$\log_{10}|K_{ij}|$  [ns$^{-1}$]")
    _apply_pub_axes(ax, title=f"Rate matrix ({len(basin_labels)} basins)")
    fig.tight_layout()
    fig.savefig(out_rates, dpi=300)

with publication_style():
    fig, ax = plt.subplots(figsize=(3.3, 2.8))
    bars = ax.bar(
        range(len(basin_labels)),
        exit_times_ns.to_numpy(),
        color=[colour_for(index) for index in range(len(basin_labels))],
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_yscale("log")
    ax.set_xticks(range(len(basin_labels)))
    ax.set_xticklabels(basin_labels)
    for bar, value in zip(bars, exit_times_ns.to_numpy()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.1f}",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    _apply_pub_axes(ax, xlabel="Basin", ylabel="Exit time  [ns]", title="Mean exit times")
    fig.tight_layout()
    fig.savefig(out_exit, dpi=300)

print(f"Saved {out_paths.relative_to(ROOT)}")
print(f"Saved {out_profiles.relative_to(ROOT)}")
print(f"Saved {out_rates.relative_to(ROOT)}")
print(f"Saved {out_exit.relative_to(ROOT)}")
plt.show()


In [ ]:
exit_times_ns
